# Module 2 Assignment: Salary Function

**Purpose:** For this assignment, I worked with the San Francisco employee salary dataset to build a simple employee information lookup system using Python. The program allows a user to search for an employee by name, retrieve their salary details, and organise the records using a Python dictionary. I also included error handling to manage common problems, such as missing files or unsuccessful employee searches. Finally, the selected employee's information is exported to a CSV file, compressed into a ZIP archive, and accessed again using R.

**Dataset expected:** The analysis uses Total.csv, which is the salary dataset referenced in the original sf-salary.ipynb notebook. The CSV file is stored in the same project folder as the completed notebook so that it can be imported directly during execution.

**Important data note:** While working with the dataset, I noted that some employee names appear across multiple years. To avoid losing information, I designed the lookup function to return all available records matching the employee's name instead of selecting just one year's record. This allows the program to present a more complete picture of the employee's salary history.

## 1. Import the required Python libraries

For this project, I used pandas to load and work with the salary records, while pathlib helped me organise file locations and manage the output folders. I used Python's built-in zipfile module to package the exported employee details into a ZIP archive. This meant I did not need to install any extra library for the file compression process.

In [7]:
from pathlib import Path
import zipfile
import pandas as pd
import sys

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Import the salary data

The notebook is set up to read the salary records from a file named `Total.csv`, which I placed in the same folder as the notebook. I also included checks to identify possible file-related issues and display a clear error message if the dataset is missing or cannot be accessed properly.

In [8]:
DATA_FILE = Path("Total.csv")

try:
    if not DATA_FILE.is_file():
        raise FileNotFoundError(
            "Total.csv was not found. Place Total.csv in the same folder as this notebook, "
            "or change DATA_FILE to the correct file path."
        )

    df = pd.read_csv(DATA_FILE, low_memory=False)
    print("Salary data imported successfully.")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")

except FileNotFoundError as error:
    print(f"File error: {error}")
    raise
except pd.errors.ParserError as error:
    print(f"CSV format error: {error}")
    raise
except (PermissionError, OSError) as error:
    print(f"File access error: {error}")
    raise

Salary data imported successfully.
Rows: 312,882
Columns: 9


## 3. Inspect and prepare the data

The original notebook contains employee information such as `EmployeeName` and `JobTitle`, together with different salary components, including `BasePay`, `OvertimePay`, `OtherPay`, `Benefits`, `TotalPay`, and `TotalPayBenefits`. Before working with the records, I check that the necessary columns are available. I also convert the salary fields into numeric format so they can be processed correctly. Where the dataset contains entries such as `Not Provided`, these are treated as missing values (`NaN`) instead of causing the program to stop.

In [9]:
required_columns = ["EmployeeName"]
missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise KeyError(
        "The salary file is missing required column(s): " + ", ".join(missing_columns)
    )

salary_columns = [
    "BasePay", "OvertimePay", "OtherPay", "Benefits",
    "TotalPay", "TotalPayBenefits"
]

for column in salary_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (312882, 9)


,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year
0,NATHANIEL FORD,GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY,167411.18,0.00,400184.25,NaN,567595.43,567595.43,2011
1,GARY JIMENEZ,CAPTAIN III (POLICE DEPARTMENT),155966.02,245131.88,137811.38,NaN,538909.28,538909.28,2011
2,ALBERT PARDINI,CAPTAIN III (POLICE DEPARTMENT),212739.13,106088.18,16452.60,NaN,335279.91,335279.91,2011
3,CHRISTOPHER CHONG,WIRE ROPE CABLE MAINTENANCE MECHANIC,77916.00,56120.71,198306.90,NaN,332343.61,332343.61,2011
4,PATRICK GARDNER,"DEPUTY CHIEF OF DEPARTMENT,(FIRE DEPARTMENT)",134401.60,9737.00,182234.59,NaN,326373.19,326373.19,2011


## 4. Process employee records using a Python dictionary

The dictionary uses a normalised employee name (trimmed and converted to lowercase) as its key. Each key stores a list of matching employee records. This design handles names that occur in multiple years without overwriting earlier records.

In [10]:
employee_dictionary = {}

for record in df.to_dict(orient="records"):
    employee_name = record.get("EmployeeName")

    # Skip records where the employee name is missing.
    if pd.isna(employee_name):
        continue

    dictionary_key = str(employee_name).strip().casefold()
    if dictionary_key == "":
        continue

    employee_dictionary.setdefault(dictionary_key, []).append(record)

print(f"Unique employee names stored in dictionary: {len(employee_dictionary):,}")

Unique employee names stored in dictionary: 99,865


## 5. Create a function to retrieve an employee's details

`get_employee_details()` accepts an employee name and returns a DataFrame containing all rows for that employee. The search is case-insensitive, so `NATHANIEL FORD` and `Nathaniel Ford` are treated as the same name.

In [11]:
def get_employee_details(employee_name):
    """Return all salary records for an employee name."""
    if not isinstance(employee_name, str) or not employee_name.strip():
        raise ValueError("Please enter a valid, non-empty employee name.")

    dictionary_key = employee_name.strip().casefold()

    if dictionary_key not in employee_dictionary:
        raise KeyError(f"No employee was found with the name: {employee_name}")

    return pd.DataFrame(employee_dictionary[dictionary_key])


# Example function call using a name shown in the starter notebook:
# get_employee_details("NATHANIEL FORD")
print("Employee lookup function created successfully.")

Employee lookup function created successfully.


## 6. Export employee details to CSV and create the ZIP archive

When the program runs, it asks me to enter the employee's name. It then searches the records and retrieves all entries matching that name. The selected information is saved in a CSV file called `Employee_Details.csv`, which is placed inside the `Employee Profile` folder. After saving the file, the program creates `Employee Profile.zip` containing the folder and its CSV file. I retained the folder structure in the archive to ensure that the R script can locate and extract the file correctly.

In [12]:
from pathlib import Path
import zipfile
import pandas as pd
from IPython.display import display


def export_employee_profile(employee_details, output_directory=None):
    """Save employee details to CSV and package the Employee Profile folder."""

    if not isinstance(employee_details, pd.DataFrame) or employee_details.empty:
        raise ValueError("There are no employee details to export.")

    if output_directory is None:
        output_directory = Path.cwd()

    output_directory = Path(output_directory)

    profile_folder = output_directory / "Employee Profile"
    profile_folder.mkdir(parents=True, exist_ok=True)

    csv_path = profile_folder / "Employee_Details.csv"
    zip_path = output_directory / "Employee Profile.zip"

    employee_details.to_csv(csv_path, index=False)

    # Create the ZIP archive
    with zipfile.ZipFile(
        zip_path,
        mode="w",
        compression=zipfile.ZIP_DEFLATED
    ) as zip_file:
        zip_file.write(
            csv_path,
            arcname="Employee Profile/Employee_Details.csv"
        )

    return csv_path, zip_path


# Request employee details and export them
try:
    requested_name = input("Enter the employee's full name: ").strip()

    selected_employee = get_employee_details(requested_name)

    print("Employee details found:")
    display(selected_employee)

    csv_file, zip_file = export_employee_profile(selected_employee)

    print(f"CSV file saved to: {csv_file.resolve()}")
    print(f"ZIP archive created: {zip_file.resolve()}")

except ValueError as error:
    print(f"Input error: {error}")

except KeyError as error:
    print(f"Lookup error: {error}")

except PermissionError as error:
    print(f"Permission error: {error}")

except OSError as error:
    print(f"File handling error: {error}")

Employee details found:


,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year
0,NATHANIEL FORD,GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY,167411.18,0.0,400184.25,NaN,567595.43,567595.43,2011


CSV file saved to: C:\Users\SAIDU T.H KAMARA\OneDrive - Emory\Pictures\NEXTFORD UNIVERSITY\COURSE 2\Python Module 2\Employee Profile\Employee_Details.csv
ZIP archive created: C:\Users\SAIDU T.H KAMARA\OneDrive - Emory\Pictures\NEXTFORD UNIVERSITY\COURSE 2\Python Module 2\Employee Profile.zip


## 7. Verify the exported files

Run this cell after a successful employee lookup and export. It checks that the CSV and ZIP archive exist and lists the files stored inside the archive.

In [13]:
zip_path = Path("Employee Profile.zip")

try:
    if not zip_path.is_file():
        raise FileNotFoundError(
            "Employee Profile.zip was not created. Run the export cell successfully first."
        )

    with zipfile.ZipFile(zip_path, mode="r") as zip_file:
        print("Files inside the ZIP archive:")
        for file_name in zip_file.namelist():
            print("-", file_name)
        bad_file = zip_file.testzip()
        if bad_file is not None:
            raise zipfile.BadZipFile(f"The ZIP archive contains a damaged file: {bad_file}")

    print("ZIP archive verification completed successfully.")

except (FileNotFoundError, zipfile.BadZipFile, OSError) as error:
    print(f"ZIP verification error: {error}")
    raise

Files inside the ZIP archive:
- Employee Profile/Employee_Details.csv
ZIP archive verification completed successfully.


## 8. R Component: Extracting and Viewing Employee Data: I Used R to unzip the folder and display the employee data

I completed the R part of this assignment using a separate script named unzip_employee_profile.R. After generating the ZIP archive in Python, I opened the R script in RStudio and ran it to extract the employee files, read the CSV dataset, and display the selected employee's information.

In [ ]:
# R code is provided in the companion file: unzip_employee_profile.R
# Open and run that script in RStudio after completing the Python export step.

## 9. Assignment requirements checklist

To ensure that all parts of the assignment have been completed, I have checked that my solution covers the following tasks:
- Imported the provided salary dataset (Total.csv) into the Jupyter Notebook for analysis.
- Developed a Python function that allows users to search for an employee by name and retrieve the corresponding salary information.
- Used a Python dictionary to organise employee records and make it easier to find individual employee details.
- Added error-handling mechanisms to deal with issues such as missing files, incorrect employee names, invalid inputs, and file access problems.
- Implemented a process for exporting the selected employee's information to a CSV file, saving it in an Employee Profile folder, and compressing the folder into Employee Profile.zip.
- Created an R script that extracts the ZIP archive, reads the exported CSV file, and displays the employee information in RStudio

